# 01 · Ingestion — what is in our knowledge base?

**Step 1 of the Study Assistant project.** This notebook loads every course file and checks that the text we extracted is complete and clean *before* we build any retrieval on top of it (garbage in → garbage out).

What the ingestion does (code in `src/study_assistant/ingest/`):

| Source | Unit (one `Document`) | Notes |
|---|---|---|
| Lecture slides `.pptx` | one slide | title, text boxes, tables, speaker notes; slide-number fields removed |
| Practical / lab notebooks `.ipynb` | one *section*: an explanation + the code that follows it | outputs and `pip install` lines dropped |
| Activity / guide PDFs | one page | PyMuPDF text in reading order; headers, footers, URLs, page counters removed |
| Image-only PDF (`module10_rl_slides.pdf`) | one slide | OCR with RapidOCR, cached; layout rebuilt so side-by-side boxes don't interleave |

Every document keeps metadata for citations: **file title, lecture number, slide/page/cell locator**.

> Run `uv run python -m study_assistant.ingest` once to (re)build `data/processed/documents.jsonl`. The cell below rebuilds it if it is missing.

In [1]:
import pandas as pd
from study_assistant.config import COURSE_DIR, DOCUMENTS_PATH
from study_assistant.ingest import load_corpus, save_documents, load_documents

if not DOCUMENTS_PATH.exists():
    save_documents(load_corpus(COURSE_DIR))
docs = load_documents()

df = pd.DataFrame([{**d.to_dict(), **d.extra, "chars": len(d.text)} for d in docs])
df["lecture"] = df["lecture"].astype("Int64")
print(f"{len(df)} documents from {df.source_file.nunique()} files  (course dir: {COURSE_DIR.name})")
df[["doc_id", "source_type", "lecture", "section", "chars"]].head()

507 documents from 47 files  (course dir: Class Files)


,doc_id,source_type,lecture,section,chars
0,Agentic_workflow.ipynb::section-1,notebook,<NA>,Let us import necessary libraries,121
1,Agentic_workflow.ipynb::section-2,notebook,<NA>,LangGraph is a flowchart for AI agents — you c...,430
2,Agentic_workflow.ipynb::section-3,notebook,<NA>,Create Nodes,687
3,Agentic_workflow.ipynb::section-4,notebook,<NA>,Build Graph,378
4,Agentic_workflow.ipynb::section-5,notebook,<NA>,Visualize the graph,152


## 1. What did we load, per file?
Slide counts should equal the number of slides in each deck (this is also asserted in `tests/test_corpus.py`).

In [2]:
per_file = (df.groupby("source_file")
              .agg(type=("source_type", "first"), lecture=("lecture", "first"), units=("doc_id", "count"),
                   chars=("chars", "sum"), low_content=("low_content", "sum"))
              .sort_values(["lecture", "type"], na_position="last"))
pd.set_option("display.max_rows", 100)
per_file

,type,lecture,units,chars,low_content
source_file,,,,,
Module 1 Custom Setup Guide-3.pdf,guide,1,4,10320,0
Module 1 - Introduction-1.pptx,slides,1,31,12054,6
Module_2_Practical_1_Probability-1.ipynb,notebook,2,3,9400,0
Module_2_Practical_2_Word_Sampling-1.ipynb,notebook,2,5,10308,0
Module_2_Practical_3_Word_Embeddings-2.ipynb,notebook,2,9,3576,0
Module 2 - Basics of Generative AI.pptx,slides,2,25,8813,5
Module 3 Activity - First Docker_FastAPI Project Setup and Simple Text Generator-1.pdf,activity,3,1,4715,0
M3_Prompting_Strategies_colab.ipynb,notebook,3,1,4749,0
Module 3 - Applications of Generative AI.pptx,slides,3,24,9904,6


## 2. Coverage per lecture
Each lecture should have slides, and most have a practical notebook and/or activity. Lecture = `<NA>` are the agents / LLM add-on labs (RAG, LangGraph, evals, fine-tuning), which are not tied to one lecture deck.

Note the **lecture mapping**: practical notebooks are numbered differently from lectures (e.g. `Module_4_Practical_VAE.ipynb` is the practical for the **Lecture 5** VAE slides). `ingest/catalog.py` maps each notebook to the lecture whose content it covers, so citations point students to the right lecture.

In [3]:
coverage = pd.crosstab(df["lecture"].fillna(-1).astype(int).replace(-1, "add-on"), df["source_type"], margins=True)
coverage

source_type,activity,guide,notebook,slides,All
lecture,,,,,
1,0,4,0,31,35
2,0,0,17,25,42
3,1,0,1,24,26
4,3,0,29,23,55
5,2,0,16,17,35
6,3,0,10,21,34
7,0,0,0,10,10
8,2,0,16,15,33
9,0,0,19,17,36


## 3. How long are the documents?
This tells us how to chunk in Step 2. The embedding model we plan to use (`all-MiniLM-L6-v2`) only reads the first **256 word-pieces** (~1,000 characters) of a text, so long units have to be split, or their tail would be invisible to retrieval.

In [4]:
df.groupby("source_type")["chars"].describe(percentiles=[.5, .9]).round(0)

,count,mean,std,min,50%,90%,max
source_type,,,,,,,
activity,11.0,1067.0,1236.0,198.0,814.0,1014.0,4715.0
guide,4.0,2580.0,462.0,1972.0,2668.0,2964.0,3011.0
notebook,250.0,1961.0,1779.0,25.0,1458.0,4091.0,15774.0
slides,242.0,391.0,258.0,12.0,407.0,713.0,1057.0


In [5]:
long_units = df[df.chars > 1000]
print(f"{len(long_units)} of {len(df)} documents are longer than ~1000 chars and will need splitting in Step 2")
long_units.groupby("source_type").size()

171 of 507 documents are longer than ~1000 chars and will need splitting in Step 2


source_type
activity      2
guide         4
notebook    164
slides        1
dtype: int64

## 4. Low-content documents
Title slides and section dividers (e.g. a slide that only says *"Variational Autoencoder"*) have no explanatory text. They stay in `documents.jsonl` but are flagged `low_content` so the index can skip them — otherwise they would out-rank the slides that actually explain the topic.

In [6]:
df[df.low_content][["doc_id", "text"]].assign(text=lambda x: x.text.str.replace("\n", " / ").str[:70])

,doc_id,text
9,Module 1 - Introduction-1.pptx::slide-1,Module 1: Introduction / / Applied Generative...
12,Module 1 - Introduction-1.pptx::slide-4,Introduction and Applications
24,Module 1 - Introduction-1.pptx::slide-16,Some Examples
33,Module 1 - Introduction-1.pptx::slide-25,Course Overview
37,Module 1 - Introduction-1.pptx::slide-29,Syllabus Walkthrough / / Go over the syllabus...
39,Module 1 - Introduction-1.pptx::slide-31,Coding Environment Setup / / Custom Setup Guide
44,Module 10 - Basics of Reinforcement Learning.p...,Module 10: Basics of Reinforcement Learning / ...
46,Module 10 - Basics of Reinforcement Learning.p...,Motivation and Components of an RL Problem
53,Module 10 - Basics of Reinforcement Learning.p...,Reinforcement Learning Problem Formulation
58,Module 11 - Application of Reinforcement Learn...,Module 11: Application of Reinforcement Learni...


## 5. Spot checks — one example of each kind
Compare these against the original files to confirm nothing important is lost.

In [7]:
def show(doc_id, n=900):
    d = next(d for d in docs if d.doc_id == doc_id)
    print(f"[{d.citation}]  lecture={d.lecture}  section={d.section!r}  extra={d.extra}\n")
    print(d.text[:n])

show("Module 5 - Modern Deep Learning Architectures Part 2.pptx::slide-16")

[Module 5 – Modern Deep Learning Architectures Part 2, slide 16]  lecture=5  section='New Loss Function'  extra={'n_images': 1, 'has_notes': False, 'n_slides': 17, 'low_content': False}

New Loss Function

Practical:
Variational Autoencoder

The Kullback-Leibler (KL) Divergence quantifies how much one probability distribution differs from another.
In our case it corresponds to how much the learned latent distribution differs from a standard normal distribution and can be shown to have a special form:
-0.5 * sum(1 + z_log_var - z_mean ^ 2 - exp(z_log_var))
The final loss function that we use in training VAE takes the form:


In [8]:
show("Module_4_Practical_VAE.ipynb::section-6")

[Module 4: Practical - Variational Autoencoders, cells 15-17 (Autoencoder Architecture)]  lecture=5  section='Autoencoder Architecture'  extra={'has_code': True, 'low_content': False}

## Autoencoder Architecture

We can now define our Encoder architecture, which consists of three convolutional layers followed by a fully connected layer.

```python
class Encoder(nn.Module):
    def __init__(self):
        super(Encoder, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(128 * 4 * 4, 2)  # Adjusted for input size (32,32)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = self.flatten(x)
        x = self.fc(x)
        return x
```


In [9]:
show("Module 5 Activity - Adding VAE to Helper Library.pdf::page-1", 700)

[Module 5 Activity – Adding VAE to Helper Library, page 1]  lecture=5  section='Variational Autoencoder'  extra={'ocr': False, 'n_pages': 2, 'low_content': False}

Variational Autoencoder

As part of this activity we will add the variational autoencoder to the helper library we developed as

part of Module 4.

You will encorporate appropriate code from Module5-VAE notebook into your helper_lib module.

1. Model Module

In model.py , define your neural network model:

import torch.nn as nn

def get_model(model_name):
# TODO: define and return the appropriate model_name - one of: FCNN, CNN,
EnhancedCNN, VAE
return model

2. Trainer Module

In trainer.py , add train_vae_model function:

import torch

def train_vae_model(model, data_loader, criterion, optimizer, device='cpu',
epochs=10):
# TODO: run several iterations of the training loop (based on epochs



In [10]:
# OCR'd slide: the two callout boxes at the bottom are read one after the other, not interleaved
show("module10_rl_slides.pdf::slide-12")

[Module 10 – Basics of Reinforcement Learning (extended deck), slide 12]  lecture=10  section='Unrolling the loop'  extra={'ocr': True, 'n_pages': 28, 'low_content': False}

TRAJECTORY
Unrolling the loop
At time t the state is s(t), the agent takes action a(t) and gets reward r(t). The action changes the
environment — and that part is not under our control. The environment evolves by some unknown function
f:
s(t+1) = f( s(t), a(t) )
Repeating the choose-action-then-update loop gives a sequence called a trajectory — also called an
episode or a rollout. All three words mean the same thing.
τ = (so,a0,r1,s1,a1,r2,s2,...)
We assume discrete time
t = 0, 1, 2, . In the continuous case the dynamics are described by a
differential equation instead — same ideas, harder maths.
f is unknown — and that's the point
If you knew f exactly you could just plan (that's what Dijkstra did in the
notebook). RL exists precisely because you usually don't.


## 6. Known limitations (to mention in the report)

1. **Equations and diagrams that are images are not captured.** e.g. Lecture 5 slide 16 says *"The final loss function that we use in training VAE takes the form:"* and the formula itself is a picture. `extra.n_images` records how many pictures a slide has. A possible later improvement is captioning those images with a vision model.
2. **OCR is not perfect**: subscripts/Greek letters in formulas come out approximately (`τ = (so,a0,r1,...)`), and small UI labels can be dropped.
3. **Code indentation in PDFs is lost** (PyMuPDF flattens it). Fine for retrieval and explanation; the notebooks keep the exact code.
4. **Notebook outputs are dropped** (training logs, printed tensors). They rarely explain concepts and would add noise.
5. **Students' own assignment submissions are deliberately excluded**: the assistant answers from what was *taught*.